In [11]:
import simpy
import random

# Параметры моделирования
RANDOM_SEED = 42
PALLET_ARRIVAL_INTERVAL = 5  # Поддоны прибывают каждые 5 минут
STORAGE_TIME_MEAN = 20       # Среднее время хранения поддона
FORKLIFT_COUNT = 2           # Количество автопогрузчиков

def pallet_process(env, name, forklift):
    """Процесс для одного поддона."""
    print(f'{env.now:.2f}: Поддон {name} прибыл в цех.')

    # Запрашиваем автопогрузчик
    with forklift.request() as request:
        yield request
        print(f'{env.now:.2f}: Поддон {name} захватил автопогрузчик.')

        # Перемещение на склад (моделируется задержкой)
        yield env.timeout(2)
        print(f'{env.now:.2f}: Поддон {name} доставлен на склад.')

    # Хранение на складе
    storage_time = random.expovariate(1.0 / STORAGE_TIME_MEAN)
    print(f'{env.now:.2f}: Поддон {name} будет храниться {storage_time:.2f} минут.')
    yield env.timeout(storage_time)

    print(f'{env.now:.2f}: Поддон {name} покинул склад и систему.')


def pallet_source(env, forklift):
    """Генерирует поддоны."""
    pallet_id = 0
    while True:
        # Ожидаем прибытия следующего поддона
        yield env.timeout(PALLET_ARRIVAL_INTERVAL)
        pallet_id += 1
        # Создаем новый процесс для прибывшего поддона
        env.process(pallet_process(env, f'Поддон-{pallet_id}', forklift))


# Настройка и запуск симуляции
print('Запуск модели заводского цеха')
random.seed(RANDOM_SEED)

# Создаем среду моделирования
env = simpy.Environment()

# Создаем ресурсы (автопогрузчики)
forklift_resource = simpy.Resource(env, capacity=FORKLIFT_COUNT)

# Запускаем процесс-генератор поддонов
env.process(pallet_source(env, forklift_resource))

# Запускаем симуляцию на определенное время (например, N минут)
env.run(until=600)

print('Моделирование завершено.')

Запуск модели заводского цеха
5.00: Поддон Поддон-1 прибыл в цех.
5.00: Поддон Поддон-1 захватил автопогрузчик.
7.00: Поддон Поддон-1 доставлен на склад.
7.00: Поддон Поддон-1 будет храниться 20.40 минут.
10.00: Поддон Поддон-2 прибыл в цех.
10.00: Поддон Поддон-2 захватил автопогрузчик.
12.00: Поддон Поддон-2 доставлен на склад.
12.00: Поддон Поддон-2 будет храниться 0.51 минут.
12.51: Поддон Поддон-2 покинул склад и систему.
15.00: Поддон Поддон-3 прибыл в цех.
15.00: Поддон Поддон-3 захватил автопогрузчик.
17.00: Поддон Поддон-3 доставлен на склад.
17.00: Поддон Поддон-3 будет храниться 6.43 минут.
20.00: Поддон Поддон-4 прибыл в цех.
20.00: Поддон Поддон-4 захватил автопогрузчик.
22.00: Поддон Поддон-4 доставлен на склад.
22.00: Поддон Поддон-4 будет храниться 5.05 минут.
23.43: Поддон Поддон-3 покинул склад и систему.
25.00: Поддон Поддон-5 прибыл в цех.
25.00: Поддон Поддон-5 захватил автопогрузчик.
27.00: Поддон Поддон-5 доставлен на склад.
27.00: Поддон Поддон-5 будет храниться

Предназначение дискретно-событийной модели, реализованной на языке Python с помощью библиотеки SimPy, заключается в имитации работы сложной системы как последовательности конкретных событий во времени. Такая модель нужна для анализа динамики процессов, где состояние системы меняется не непрерывно, а в моменты завершения определенных операций — например, когда фура приехала на разгрузку или станок закончил обработку детали. Это позволяет проводить эксперименты в цифровой среде, находить «узкие места» (bottlenecks) и тестировать гипотезы по улучшению логистики без реальных затрат и рисков для производства.


Обязательными составляющими для формализации в такой модели являются сущности (агенты), ресурсы и события. В коде на Python сущности — это объекты или процессы (поддоны, фуры), которые имеют свои жизненные циклы. Ресурсы — это объекты с ограниченной емкостью (автопогрузчики, станки с ЧПУ), за которые сущности конкурируют в процессе выполнения модели. Также необходимо формализовать логику очередей, временные задержки (timeouts), описывающие длительность операций, и системное время, которое управляется средой моделирования.


С точки зрения оптимизации, данная модель описывает множество альтернатив, которые мы можем сравнивать между собой — например, разное количество техники или разные графики поставок. На это множество накладываются ограничения, такие как лимит доступных ресурсов, физическая пропускная способность цеха и временные рамки смен. Критерием для сравнения этих альтернатив и выбора лучшего решения в модели выступают показатели эффективности: минимизация времени ожидания в очередях, максимизация загрузки оборудования или общее количество выпущенной продукции за заданный период.